In [1]:
from ingest import load_faq_data

documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)
        

161

In [3]:
documents = documents_llm

In [6]:
doc = documents[0]
print(doc["id"])
print(doc["course"])
print(doc["section"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
llm-zoomcamp
General Course-Related Questions
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [7]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [8]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [10]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [11]:
import json

user_prompt = json.dumps(doc)

In [12]:
from evaluation_utils import llm_structured
from evaluation_utils import calc_price

In [13]:
result, answer_usage= llm_structured(
    openai_client, 
    data_gen_instructions,
    user_prompt,
    Questions
)

In [14]:
answer_usage
calc_price(answer_usage)

{'input_cost': 0.00015525, 'output_cost': 0.0004815, 'total_cost': 0.00063675}

In [15]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course late — can I still enroll, or is it too late to join?',
  'document': '74eb249bbf'},
 {'question': 'If I join now, do I still have a shot at getting the certificate, or did I miss the deadline?',
  'document': '74eb249bbf'},
 {'question': 'Is it okay to start the course after it has already begun?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do if I want a certificate after joining late?',
  'document': '74eb249bbf'},
 {'question': 'Are late joiners allowed, and is submitting the project on time the only requirement for the certificate?',
  'document': '74eb249bbf'}]

In [16]:
import pandas as pd

In [17]:
pd.DataFrame(records)

,question,document
0,I just found this course late — can I still en...,74eb249bbf
1,"If I join now, do I still have a shot at getti...",74eb249bbf
2,Is it okay to start the course after it has al...,74eb249bbf
3,What do I need to do if I want a certificate a...,74eb249bbf
4,"Are late joiners allowed, and is submitting th...",74eb249bbf


In [18]:
from evaluation_utils import llm_structured_retry

In [19]:
def generate_ground_truth(doc):

    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc['id']
        })
    return results, usage
                         

In [20]:
generate_ground_truth(doc)

([{'question': 'Can I still join the course if I just found it now, or is it too late?',
   'document': '74eb249bbf'},
  {'question': 'If I join late, am I still eligible for a certificate?',
   'document': '74eb249bbf'},
  {'question': "What do I need to do to get a certificate if I'm starting the course after it already began?",
   'document': '74eb249bbf'},
  {'question': 'Is it okay to start the course now and still complete the project for certification?',
   'document': '74eb249bbf'},
  {'question': 'Do I have to submit the project before submissions close in order to get the certificate?',
   'document': '74eb249bbf'}],
 ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 0}), output_tokens=101, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=308))

In [21]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [22]:
print(ground_truth)

[{'question': 'I just found this course late — can I still join and start from here?', 'document': '74eb249bbf'}, {'question': 'Am I too late to enroll, or is it still okay to take the course now?', 'document': '74eb249bbf'}, {'question': 'If I join after the course has already started, can I still get the certificate?', 'document': '74eb249bbf'}, {'question': 'What do I need to do to be eligible for the certificate if I start late?', 'document': '74eb249bbf'}, {'question': 'Is it fine to follow the course now, or do I have to submit the project before submissions close?', 'document': '74eb249bbf'}, {'question': 'I signed up for the LLM Zoomcamp, but I never got a confirmation email — is that a problem?', 'document': '977bf7786c'}, {'question': 'Do I actually need to wait for an acceptance or confirmation message before starting the course?', 'document': '977bf7786c'}, {'question': 'Can I begin watching the lessons and submitting homework even if I didn’t register first?', 'document': 

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/161 [00:00<?, ?it/s]

In [40]:
results


[([{'question': 'Can I still join the course if I just found out about it?',
    'document': '74eb249bbf'},
   {'question': 'Is it too late to start the course now?',
    'document': '74eb249bbf'},
   {'question': 'If I join late, can I still get a certificate?',
    'document': '74eb249bbf'},
   {'question': 'Do I need to submit my project before submissions close to get a certificate?',
    'document': '74eb249bbf'},
   {'question': 'What should I do if I want the certificate after joining the course late?',
    'document': '74eb249bbf'}],
  ResponseUsage(input_tokens=207, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0, cache_write_tokens_details={'ttl_12h_tokens': 0, 'ttl_30m_tokens': 0}), output_tokens=82, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=289)),
 ([{'question': 'Do I actually need to wait for a confirmation email before starting the LLM Zoomcamp?',
    'document': '977bf7786c'},
   {'question': 'If I register

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

805

In [26]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12580049999999993

In [27]:
from evaluation_utils import calc_total_price
calc_total_price(usages)

0.12580049999999993

In [28]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [46]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [29]:
len(df_ground_truth)

805